In [1]:
### ARTS as the line-by-line code
import os
# set global variables BEFORE importing ARTS
os.environ['ARTS_DATA_PATH'] = '../../arts-cat-data-2.6.14:../../arts-xml-data-2.6.14'

import pyarts
from pyarts.arts import convert
# the fluxsim module
import FluxSimulator as fsm
import seaborn as sns


# helper packages
import numpy as np
import xarray as xr
from scipy.constants import speed_of_light as c
import matplotlib.pyplot as plt

import typhon
import typhon.constants as tpc
import pandas as pd

from scipy.optimize import minimize

from cycler import cycler

In [2]:
ddq_sw = xr.open_dataset('../DDQ configurations/DDQ_SW.h5', engine = 'netcdf4')
ddq_sw


getfattr: /homedata/pczarnec/ddq-implementation-sprint/DDQ: No such file or directory
getfattr: configurations/DDQ_SW.h5: No such file or directory


<xarray.Dataset> Size: 1kB
Dimensions:  (S: 64)
Coordinates:
  * S        (S) float64 512B 2.144e+03 2.411e+03 ... 4.426e+04 4.839e+04
Data variables:
    W        (S) float64 512B ...
Attributes:
    description:  64 optimized wavenumbers and weights, in the shortwave, opt...

In [3]:
import pyarts
from scipy.interpolate import interp1d
from scipy.constants import speed_of_light as c

# Your 64 frequencies (Hz)
f_grid = convert.kaycm2freq(ddq_sw.S.data)

solar = pyarts.xml.load("star/Sun/solar_spectrum_July_2008.xml")


# Frequencies in the XML (Hz)
f_sun = np.asarray(solar.grids[0])

# Solar spectrum values
I_sun = np.asarray(solar.data).squeeze()[:, 0]


interp = interp1d(
    f_sun,
    I_sun,
    kind="linear",
    bounds_error=False,
    fill_value="extrapolate",
)

I64 = interp(f_grid)

I = I64 * c * 100


In [4]:
I

array([4.56130814e+02, 5.96515400e+02, 8.38639395e+02, 1.03952574e+03,
       1.13349978e+03, 1.22572234e+03, 1.23095790e+03, 1.30562457e+03,
       1.74951997e+03, 2.22194401e+03, 2.32790741e+03, 2.91845612e+03,
       3.06323522e+03, 3.14996223e+03, 3.13945313e+03, 3.15370366e+03,
       3.14712934e+03, 3.27356259e+03, 3.29035687e+03, 3.25356664e+03,
       3.33025726e+03, 3.32942417e+03, 3.33033721e+03, 3.34325519e+03,
       3.41402413e+03, 3.38064105e+03, 3.35344029e+03, 2.68652012e+03,
       3.32579675e+03, 3.29420052e+03, 3.20261903e+03, 3.11545607e+03,
       3.12317277e+03, 3.12518723e+03, 2.89892509e+03, 2.89860478e+03,
       2.92015360e+03, 2.89680188e+03, 2.51134869e+03, 1.94270024e+03,
       2.04513492e+03, 2.04816231e+03, 1.31061360e+03, 7.61494946e+02,
       5.58044972e+02, 4.90110332e+02, 2.93565097e+02, 2.45717468e+02,
       1.63917403e+02, 1.97801807e+02, 1.11242773e+02, 3.74863556e+01,
       3.82272246e+01, 7.82107359e+01, 2.51920554e+01, 3.32655342e+01,
      

In [ ]:
np.dot(I, ddq_sw.W.data)/45882.451540092574


1361.0

In [8]:
scale_source = 45882.451540092574
I = I/scale_source

In [9]:
np.dot(I, ddq_sw.W.data)

1361.0

In [26]:
np.trapz(I, ddq_sw.S.data)

62389880.65735784

In [27]:
np.trapz(I64, f_grid)

62389880.65735784

In [29]:
ddq_sw

<xarray.Dataset> Size: 2kB
Dimensions:       (S: 64)
Coordinates:
  * S             (S) float64 512B 2.144e+03 2.411e+03 ... 4.426e+04 4.839e+04
Data variables:
    W             (S) float64 512B 1.509e+03 1.244e+03 ... 0.01592 0.01849
    solar_source  (S) float64 512B 456.1 596.5 838.6 ... 9.108 9.925 2.085
Attributes:
    description:  64 optimized wavenumbers and weights, in the shortwave, opt...

In [11]:
ddq_sw = ddq_sw.assign({"solar_source" : (("S"), I)})
ddq_sw.to_netcdf("../DDQ configurations/DDQ_SW_source.h5")

getfattr: /homedata/pczarnec/ddq-implementation-sprint/DDQ: No such file or directory
getfattr: configurations/DDQ_SW_source.h5: No such file or directory
